# VisionAssist Phase 8–9 — Adapter evaluation and 1,000-example training

This notebook is the canonical restart-safe Colab workflow after the completed 32-example overfit experiment. It restores prepared data and the best exported adapter, evaluates that adapter on train/validation/test subsets, preserves the reports, audits the next 1,000-example subset, validates one real training batch, and starts a fresh resumable smoke-training experiment only after an explicit approval gate.

Run cells in order in a **fresh GPU runtime**. The 32-example adapter is evaluated but is not used to initialize the new 1,000-example run.

## A. Runtime and project setup

In [ ]:
#@title 1. Settings and CUDA allocator
import os
from pathlib import Path

# These must be set before importing Torch.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True,garbage_collection_threshold:0.8"
)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/moshiur00/visionassist-industrial-visual-inspection.git"
REPO_BRANCH = "main"
PROJECT_ROOT = Path("/content/visionassist-industrial-visual-inspection")
DRIVE_ROOT = Path("/content/drive/MyDrive/visionassist")
DRIVE_DATA_ARCHIVE = DRIVE_ROOT / "data/visionassist_prepared_data.tar.gz"
OVERFIT_RUN_ID = "qwen25vl3b_qlora_overfit_v1"
SMOKE_RUN_ID = "qwen25vl3b_qlora_smoke_v1"
SMOKE_CONFIG = PROJECT_ROOT / "configs/training/qwen25vl3b_qlora_smoke.yaml"


In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")
for path in (DRIVE_ROOT / "data", DRIVE_ROOT / "checkpoints", DRIVE_ROOT / "outputs"):
    path.mkdir(parents=True, exist_ok=True)
assert DRIVE_DATA_ARCHIVE.is_file(), f"Missing prepared-data archive: {DRIVE_DATA_ARCHIVE}"


In [ ]:
#@title 3. Clone or update the repository
import shutil
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "uv"], check=True)
if (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
else:
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
print("Repository commit:", commit)


In [ ]:
#@title 4. Install dependencies, then verify the GPU
subprocess.run(["uv", "sync", "--extra", "training", "--extra", "dev"], cwd=PROJECT_ROOT, check=True)

import torch
assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
properties = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(properties.total_memory / 1024**3, 2))
print("BF16 supported:", torch.cuda.is_bf16_supported())
print("CUDA allocator:", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))
subprocess.run(["nvidia-smi"], check=False)


## B. Restore and validate prepared data

In [ ]:
#@title 5. Restore prepared data to fast local storage
import tarfile

required_data = [
    PROJECT_ROOT / "data/raw/visa",
    PROJECT_ROOT / "data/processed/visa_instructions/train.jsonl",
    PROJECT_ROOT / "data/processed/visa_instructions/validation.jsonl",
    PROJECT_ROOT / "data/processed/visa_instructions/test.jsonl",
    PROJECT_ROOT / "data/benchmarks/visa_baseline_v1/benchmark.jsonl",
]
if not all(path.exists() for path in required_data):
    with tarfile.open(DRIVE_DATA_ARCHIVE, "r:gz") as archive:
        archive.extractall(PROJECT_ROOT, filter="data")
assert all(path.exists() for path in required_data)
print("Prepared data ready.")


In [ ]:
#@title 6. Normalize all legacy Windows image paths
import hashlib
import json
from pathlib import PurePosixPath

MARKER = ("data", "raw", "visa")

def project_relative_image_path(value: str) -> str:
    normalized = value.replace("\\", "/")
    parts = PurePosixPath(normalized).parts
    lowered = tuple(part.lower() for part in parts)
    for index in range(len(parts) - len(MARKER) + 1):
        if lowered[index:index + len(MARKER)] == MARKER:
            return PurePosixPath(*parts[index:]).as_posix()
    candidate = PurePosixPath(normalized)
    if not candidate.is_absolute() and ".." not in candidate.parts:
        return candidate.as_posix()
    raise ValueError(f"Cannot normalize image path: {value}")

def normalize_jsonl(path: Path) -> tuple[int, int]:
    records, changed_records, changed_paths = [], 0, 0
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            record = json.loads(line)
            changed = False
            for message in record.get("messages", []):
                if message.get("role") != "user":
                    continue
                for item in message.get("content", []):
                    if item.get("type") == "image":
                        original = str(item.get("image", ""))
                        normalized = project_relative_image_path(original)
                        if normalized != original:
                            item["image"] = normalized
                            changed_paths += 1
                            changed = True
            changed_records += int(changed)
            records.append(record)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="\n") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    temporary.replace(path)
    return changed_records, changed_paths

instruction_root = PROJECT_ROOT / "data/processed/visa_instructions"
for split_name in ("train", "validation", "test"):
    path = instruction_root / f"{split_name}.jsonl"
    print(split_name, normalize_jsonl(path))

benchmark = PROJECT_ROOT / "data/benchmarks/visa_baseline_v1/benchmark.jsonl"
changed = normalize_jsonl(benchmark)
digest = hashlib.sha256(benchmark.read_bytes()).hexdigest()
manifest_path = benchmark.parent / "benchmark_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest["benchmark_sha256"] = digest
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
(benchmark.parent / "benchmark_sha256.txt").write_text(digest + "\n", encoding="utf-8")
print("benchmark", changed, digest)


In [ ]:
#@title 7. Verify every instruction path and run focused tests
for split_name in ("train", "validation", "test"):
    path = instruction_root / f"{split_name}.jsonl"
    invalid = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            image = next(item["image"] for message in record["messages"] if message["role"] == "user" for item in message["content"] if item["type"] == "image")
            if not image.startswith("data/raw/visa/") or not (PROJECT_ROOT / image).is_file():
                invalid.append((line_number, image))
    print(split_name, "invalid paths:", len(invalid))
    assert not invalid, invalid[:3]

subprocess.run(["uv", "run", "pytest", "tests/test_phase7c_inference.py", "tests/test_phase8_training.py"], cwd=PROJECT_ROOT, check=True)


## C. Restore and assess the checkpoint-50 final adapter

In [ ]:
#@title 8. Restore the best exported adapter from Drive
local_overfit_run = PROJECT_ROOT / "outputs/training" / OVERFIT_RUN_ID
drive_candidates = [
    DRIVE_ROOT / "outputs/training" / OVERFIT_RUN_ID,
    DRIVE_ROOT / "checkpoints" / OVERFIT_RUN_ID,
]
drive_overfit_run = next((path for path in drive_candidates if (path / "final_adapter/adapter_model.safetensors").is_file()), None)
assert drive_overfit_run is not None, "Could not find final_adapter in Drive."
shutil.copytree(drive_overfit_run, local_overfit_run, dirs_exist_ok=True)
adapter = local_overfit_run / "final_adapter"
for name in ("adapter_config.json", "adapter_model.safetensors"):
    path = adapter / name
    assert path.is_file(), f"Missing adapter file: {path}"
    print(name, path.stat().st_size, hashlib.sha256(path.read_bytes()).hexdigest())


In [ ]:
#@title 9. Run train, validation, and held-out adapter assessments
assessment_configs = {
    "train": "configs/inference/qwen25vl3b_overfit_checkpoint50_train.yaml",
    "validation": "configs/inference/qwen25vl3b_overfit_checkpoint50_validation.yaml",
    "test": "configs/inference/qwen25vl3b_overfit_checkpoint50_test.yaml",
}
for split_name, config_path in assessment_configs.items():
    print(f"\n===== {split_name.upper()} ASSESSMENT =====")
    subprocess.run(["uv", "run", "visionassist", "evaluate-adapter", "--config", config_path], cwd=PROJECT_ROOT, check=True)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
#@title 10. Summarize and persist Phase 9 results
assessment_root = PROJECT_ROOT / "outputs/post_training/qwen25vl3b_overfit_checkpoint50"
for split_name in ("train", "validation", "test"):
    run_dir = assessment_root / split_name
    summary = json.loads((run_dir / "assessment_summary.json").read_text(encoding="utf-8"))
    metrics = json.loads((run_dir / "evaluation/metrics.json").read_text(encoding="utf-8"))
    print(f"\n===== {split_name.upper()} =====")
    print("records:", summary["records"], "failures:", summary["failure_records"], "failure_rate:", summary["failure_rate"])
    print("failure tags:", metrics.get("failure_tag_counts", {}))
    for task, values in metrics.get("per_task", {}).items():
        compact = {key: value for key, value in values.items() if key not in {"per_label", "confusion_matrix"}}
        print(task, compact)

drive_assessment = DRIVE_ROOT / "outputs/post_training/qwen25vl3b_overfit_checkpoint50"
shutil.copytree(assessment_root, drive_assessment, dirs_exist_ok=True)
print("Saved assessment artifacts to:", drive_assessment)


## D. Prepare the fresh 1,000-example smoke run

The next run starts from the untouched base model. It must not resume from the 32-example overfit experiment.

In [ ]:
#@title 11. Audit the exact deterministic 1,000-example subset
import random
from collections import Counter

train_path = instruction_root / "train.jsonl"
records = [json.loads(line) for line in train_path.read_text(encoding="utf-8").splitlines() if line.strip()]
indices = list(range(len(records)))
random.Random(42).shuffle(indices)
selected = [records[index] for index in sorted(indices[:1000])]

conditions = Counter(row["metadata"]["condition"] for row in selected)
tasks = Counter(row["task_family"] for row in selected)
categories = Counter(row["metadata"]["category"] for row in selected)
anomalous_categories = Counter(row["metadata"]["category"] for row in selected if row["metadata"]["condition"] == "anomalous")
print("records:", len(selected), "unique images:", len({row["image_id"] for row in selected}))
print("conditions:", conditions)
print("tasks:", tasks)
print("categories:", categories)
print("anomalous categories:", anomalous_categories)
assert len(selected) == 1000
assert len(categories) == 12
assert len(tasks) == 8
assert conditions["anomalous"] > 0
assert tasks["uncertainty"] > 0


In [ ]:
#@title 12. Configure Drive checkpoint persistence
import yaml

smoke = yaml.safe_load(SMOKE_CONFIG.read_text(encoding="utf-8"))
assert smoke["run_id"] == SMOKE_RUN_ID
assert smoke["data"]["train_limit"] == 1000
smoke["checkpoints"]["persistent_output_dir"] = str(DRIVE_ROOT / "checkpoints" / SMOKE_RUN_ID)
smoke["checkpoints"]["sync_every_save"] = True
SMOKE_CONFIG.write_text(yaml.safe_dump(smoke, sort_keys=False), encoding="utf-8")
print(SMOKE_CONFIG.read_text(encoding="utf-8"))


In [ ]:
#@title 13. Inspect environment and validate one real training batch
subprocess.run(["uv", "run", "visionassist", "training-environment", "--config", str(SMOKE_CONFIG)], cwd=PROJECT_ROOT, check=True)
subprocess.run(["uv", "run", "visionassist", "training-smoke-test", "--config", str(SMOKE_CONFIG)], cwd=PROJECT_ROOT, check=True)
smoke_report = PROJECT_ROOT / "outputs/training" / SMOKE_RUN_ID / "one_batch_smoke_test.json"
report = json.loads(smoke_report.read_text(encoding="utf-8"))
print(json.dumps(report, indent=2))
assert report["passed"] is True
assert report["finite_gradients"] is True
assert report["nonzero_gradients"] is True


## E. Explicit training gate

Review the subset distribution and one-batch report above. Start training only when all assertions passed and several GiB of VRAM headroom remain. The run uses a new output directory and therefore cannot resume the old overfit optimizer state.

In [ ]:
#@title 14. Start or resume the fresh 1,000-example run
START_TRAINING = False  # Change to True only after reviewing the gate above.

if not START_TRAINING:
    raise RuntimeError("Training gate is closed. Review the audit and smoke report first.")

subprocess.run(["uv", "run", "visionassist", "train-qlora", "--config", str(SMOKE_CONFIG), "--resume", "latest"], cwd=PROJECT_ROOT, check=True)


In [ ]:
#@title 15. Inspect and persist final smoke-run artifacts
smoke_run = PROJECT_ROOT / "outputs/training" / SMOKE_RUN_ID
manifest = json.loads((smoke_run / "run_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(manifest, indent=2))
assert manifest["status"] == "completed"
assert (smoke_run / "final_adapter/adapter_model.safetensors").is_file()
drive_smoke_run = DRIVE_ROOT / "outputs/training" / SMOKE_RUN_ID
shutil.copytree(smoke_run, drive_smoke_run, dirs_exist_ok=True)
print("Saved final smoke-run artifacts to:", drive_smoke_run)


## F. Evaluate the completed 1,000-example adapter

This section evaluates the best exported smoke-run adapter on the exact 200-record validation subset and the complete frozen 2,100-record benchmark. Inference is resumable within the current runtime. Persist the completed results to Drive immediately.

In [ ]:
#@title 16. Create smoke-adapter evaluation configurations
base_config_dir = PROJECT_ROOT / "configs/inference"
generated_configs = {}
evaluation_settings = {
    "validation": {"source": "qwen25vl3b_overfit_checkpoint50_validation.yaml", "limit": 200, "seed": 43},
    "test": {"source": "qwen25vl3b_overfit_checkpoint50_test.yaml", "limit": None, "seed": 44},
}
for split_name, values in evaluation_settings.items():
    source_path = base_config_dir / values["source"]
    config = yaml.safe_load(source_path.read_text(encoding="utf-8"))
    output = f"outputs/post_training/qwen25vl3b_smoke_best/{split_name}"
    config.update({
        "run_id": f"qwen25vl3b_smoke_best_{split_name}_v1",
        "adapter_path": "outputs/training/qwen25vl3b_qlora_smoke_v1/final_adapter",
        "output_dir": output,
        "partial_predictions_path": f"{output}/predictions.partial.jsonl",
        "predictions_path": f"{output}/predictions.jsonl",
        "errors_path": f"{output}/inference_errors.jsonl",
        "run_manifest_path": f"{output}/run_manifest.json",
        "evaluation_records_path": f"{output}/evaluation_records.jsonl",
        "subset_limit": values["limit"],
        "subset_seed": values["seed"],
        "overwrite": False,
    })
    if split_name == "validation":
        config["benchmark_manifest_path"] = "outputs/training/qwen25vl3b_qlora_smoke_v1/dataset_manifest.json"
    generated_path = base_config_dir / f"qwen25vl3b_smoke_best_{split_name}.yaml"
    generated_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    generated_configs[split_name] = generated_path
    print(split_name, "->", generated_path)


In [ ]:
#@title 17. Evaluate validation and the complete frozen benchmark
for split_name in ("validation", "test"):
    print(f"\n===== {split_name.upper()} =====")
    subprocess.run(["uv", "run", "visionassist", "evaluate-adapter", "--config", str(generated_configs[split_name])], cwd=PROJECT_ROOT, check=True)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
#@title 18. Summarize and persist smoke-adapter results
smoke_assessment = PROJECT_ROOT / "outputs/post_training/qwen25vl3b_smoke_best"
for split_name in ("validation", "test"):
    run_dir = smoke_assessment / split_name
    summary = json.loads((run_dir / "assessment_summary.json").read_text(encoding="utf-8"))
    metrics = json.loads((run_dir / "evaluation/metrics.json").read_text(encoding="utf-8"))
    print(f"\n===== {split_name.upper()} =====")
    print("records:", summary["records"], "failures:", summary["failure_records"], "failure_rate:", summary["failure_rate"])
    print("failure tags:", metrics.get("failure_tag_counts", {}))
    for task, values in metrics.get("per_task", {}).items():
        compact = {key: value for key, value in values.items() if key not in {"per_label", "confusion_matrix"}}
        print(task, compact)
drive_results = DRIVE_ROOT / "outputs/post_training/qwen25vl3b_smoke_best"
shutil.copytree(smoke_assessment, drive_results, dirs_exist_ok=True)
print("Saved smoke-adapter results to:", drive_results)


In [ ]:
#@title 19. Inspect standalone defect predictions
validation_dir = smoke_assessment / "validation"
validation_records = {row["instruction_id"]: row for row in (json.loads(line) for line in (validation_dir / "evaluation_records.jsonl").read_text(encoding="utf-8").splitlines() if line.strip())}
validation_predictions = {row["instruction_id"]: row for row in (json.loads(line) for line in (validation_dir / "predictions.jsonl").read_text(encoding="utf-8").splitlines() if line.strip())}
defect_ids = [instruction_id for instruction_id, record in validation_records.items() if record["task_family"] == "defect_identification"]
for index, instruction_id in enumerate(defect_ids, start=1):
    row = validation_predictions[instruction_id]
    print(f"\n===== DEFECT {index}/{len(defect_ids)} =====")
    print("ID:", instruction_id)
    print("Ground truth:", row["ground_truth"])
    print("Prediction:", row["prediction"])


## Resume and next decision

After a disconnect, rerun setup, restore data, normalize paths, configure checkpoint persistence, and execute Cell 14 with `START_TRAINING = True`. `--resume latest` restores only the newest checkpoint belonging to `qwen25vl3b_qlora_smoke_v1`.

Do not begin the 10,000-example pilot automatically. First evaluate the best 1,000-example adapter using the same train/validation/held-out workflow, compare it with the frozen untouched-model baseline, and review category, anomaly-recall, defect, structured-report, and abstention regressions.